# Stress Index (Reproducible Notebook)

This notebook regenerates Stress Index artefacts used by the dashboard.

**Outputs**:
- `data/processed/stress_timeseries.csv`
- `data/processed/stress_global_means.csv`
- `data/processed/stress_top_windows.csv`


In [ ]:
from pathlib import Path
import os

# Resolve repository root robustly (works in GitHub/VSCode and Colab)
THIS_FILE = Path.cwd()
# If running in Colab after cloning, cwd is repo root; otherwise adjust:
if (THIS_FILE / "app.py").exists():
    REPO_ROOT = THIS_FILE
else:
    # try parent levels
    REPO_ROOT = next((p for p in THIS_FILE.parents if (p / "app.py").exists()), THIS_FILE)

DATA_DIR = REPO_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
INTERIM_DIR = DATA_DIR / "interim"
FIG_DIR = REPO_ROOT / "reports" / "figures"

for d in [PROCESSED_DIR, INTERIM_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("FIG_DIR:", FIG_DIR)

In [ ]:
import numpy as np
import pandas as pd

# Try to load existing model risk series if available; otherwise synthesize.
# These are lightweight and intended to support dashboard evaluation views.
candidates = [
    PROCESSED_DIR / "stress_timeseries.csv",
]
df = None
for p in candidates:
    if p.exists():
        df = pd.read_csv(p, parse_dates=["time"])
        print("Loaded existing stress_timeseries:", p, "rows:", len(df))
        break

if df is None or df.empty:
    print("No existing stress_timeseries.csv found — generating synthetic demo stress signals.")
    rng = np.random.default_rng(123)
    idx = pd.date_range("2021-10-25", "2021-11-01", freq="H", tz="UTC")
    df = pd.DataFrame({"time": idx})
    df["signal_loss_risk"] = rng.uniform(0, 0.6, len(idx))
    df["jamming_risk"] = rng.uniform(0, 0.5, len(idx))
    df["sla_risk"] = rng.uniform(0, 0.7, len(idx))
    df["capacity_risk"] = rng.uniform(0, 0.8, len(idx))

# Ensure timestamp
df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce")
df = df.dropna(subset=["time"])

In [ ]:
# Stress index definition: conservative max across drivers (matches dashboard logic)
drivers = ["signal_loss_risk", "jamming_risk", "sla_risk", "capacity_risk"]
for c in drivers:
    if c not in df.columns:
        df[c] = 0.0

df["stress_index"] = df[drivers].max(axis=1)

# Save time series
out_ts = PROCESSED_DIR / "stress_timeseries.csv"
df.sort_values("time").to_csv(out_ts, index=False)
print("Saved:", out_ts, "rows:", len(df))

In [ ]:
# Global means (for report / quick stats)
means = df[drivers + ["stress_index"]].mean().reset_index()
means.columns = ["metric", "mean_value"]
out_means = PROCESSED_DIR / "stress_global_means.csv"
means.to_csv(out_means, index=False)
print("Saved:", out_means)

# Top windows (for dashboard table)
top = df.sort_values("stress_index", ascending=False).head(50).copy()
out_top = PROCESSED_DIR / "stress_top_windows.csv"
top.to_csv(out_top, index=False)
print("Saved:", out_top)

## Optional plot (for quick verification)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(df["time"].dt.tz_convert("UTC"), df["stress_index"])
plt.title("Stress Index over time (demo)")
plt.xlabel("Time (UTC)")
plt.ylabel("Stress index")
plt.tight_layout()
plt.show()

## Done

You can now run the dashboard:

```bash
streamlit run app.py
```